In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s5e10/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e10/test.csv')

In [ ]:
train.info()

# feature engg

In [ ]:
for col in train.columns:
    if train[col].dtype=='O': 
        print(col,train[col].unique())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
import numpy as np

ct = ColumnTransformer([
    ('ohe',OneHotEncoder(handle_unknown='ignore'),['road_type','weather','time_of_day']),
    ('oe',OrdinalEncoder(categories=[['daylight','dim','night']],handle_unknown='use_encoded_value',unknown_value=3),['lighting'])
],remainder=StandardScaler())

In [ ]:
def feature_engg(df,training):
    
    df = df.drop(columns=['id'])
    if('accident_risk' in df.columns):
        df = df.drop(columns=['accident_risk'])
    
    columns = [col for col in df.columns if df[col].dtype==bool]
    df[columns] = df[columns].apply(lambda col:col.astype(int))

    if training:
        df = ct.fit_transform(df)
    else:
        df = ct.transform(df)

    return df

# ML Stacking

## base models

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import KFold
kf = KFold(n_splits=5,shuffle=True,random_state=3)

models = {
    'xgb':XGBRegressor(),
    'lgb':LGBMRegressor(),
    'cat':CatBoostRegressor(verbose=0)
}

oof_pred = {name: np.zeros(len(train)) for name in models.keys()}
final_model = {}

for name,model in models.items():
    for train_index,test_index in kf.split(train):
        X_train = feature_engg(train.iloc[train_index], True)
        y_train = train.iloc[train_index]['accident_risk'].values
        X_test = feature_engg(train.iloc[test_index], False)
        # y_val = train.iloc[val_index]['accident_risk'].values
        
        model.fit(X_train,y_train)
        oof_pred[name][test_index] = model.predict(X_test)

X_full = feature_engg(train,True)
y_full = train['accident_risk']

for name,model in models.items():
    model.fit(X_full,y_full)
    final_model[name] = model

## hypertuning base models

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial,model_name,cv):
    if model_name == 'xgboost':
        params = {
            'n_estimators':trial.suggest_int('n_estimators',100,300),
            'max_depth':trial.suggest_int('max_depth',10,20),
            'learning_rate':trial.suggest_float('learning_rate',0.01,0.1),
            'subsample': trial.suggest_categorical('subsample', [0.5, 0.7, 1.0]),
            'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.6, 0.8, 1.0]),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
        }
        model = XGBRegressor(device='cuda',random_state=3,**params)
        
    if model_name == 'lightgbm':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 16, 128),
            'subsample': trial.suggest_categorical('subsample', [0.5, 0.7, 1.0]),
            'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.6, 0.8, 1.0])
        }
        model = LGBMRegressor(device='gpu',random_state=3, **params)
        
    if model_name == 'catboost':
        params = {
            'iterations': trial.suggest_int('iterations', 100, 400),
            'depth': trial.suggest_int('depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 0.0, 10.0),
        }
        model = CatBoostRegressor(task_type='GPU',verbose=0,random_state=3,**params)


    score = cross_val_score(
        model, feature_engg(train,True), train['accident_risk'], cv=cv, scoring='neg_root_mean_squared_error'
    ).mean()
    
    return -score

In [ ]:
base_model_hupertuned={}
options = [
    ['xgboost',3,25],
    ['lightgbm',5,50],
    ['catboost',5,50]
]

for name,cv,n_trials in options:
    study = optuna.create_study(study_name=f'{name}',direction='minimize')
    study.optimize(
        lambda trial:objective(trial,name,cv),
        n_trials=n_trials,
        show_progress_bar=True,
        n_jobs=1
    )
    base_model_hupertuned[name]={}
    base_model_hupertuned[name]['result']=study.best_value
    base_model_hupertuned[name]['para']=study.best_trial.params

In [ ]:
print(base_model_hupertuned)

In [ ]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

xgb = XGBRegressor(
    n_estimators=194,
    max_depth=16,
    learning_rate=0.04227655262123072,
    subsample=0.5,
    colsample_bytree=0.8,
    reg_alpha=0.66057915458616,
    reg_lambda=3.01220558475813e-05,
    random_state=3,
    device='cuda'
)

cat = CatBoostRegressor(
    iterations=378,
    depth=8,
    learning_rate=0.07464147709280693,
    l2_leaf_reg=0.0767206054830952,
    random_strength=2.69832979272374,
    verbose=0,
    random_state=3,
    task_type='GPU'
)

lgb = LGBMRegressor(
    n_estimators=180,
    max_depth=15,
    learning_rate=0.04655825082477995,
    num_leaves=102,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=3,
    device='gpu'
)

In [ ]:
models = {
    'xgb':xgb,
    'lgb':lgb,
    'cat':cat
}

oof_pred = {name: np.zeros(len(train)) for name in models.keys()}
final_model = {}

for name,model in models.items():
    for train_index,test_index in kf.split(train):
        X_train = feature_engg(train.iloc[train_index], True)
        y_train = train.iloc[train_index]['accident_risk'].values
        X_test = feature_engg(train.iloc[test_index], False)
        # y_val = train.iloc[val_index]['accident_risk'].values
        
        model.fit(X_train,y_train)
        oof_pred[name][test_index] = model.predict(X_test)

X_full = feature_engg(train,True)
y_full = train['accident_risk']

for name,model in models.items():
    model.fit(X_full,y_full)
    final_model[name] = model

## meta model

In [ ]:
from sklearn.metrics import mean_squared_error,r2_score

def calculate(pred,res):
    print('rmse is -> ',np.sqrt(mean_squared_error(res,pred)))
    print('r2 is   -> ',r2_score(res,pred))

In [ ]:
meta_model_df = pd.concat([pd.DataFrame(oof_pred),pd.DataFrame(train['accident_risk'])],axis=True)

In [ ]:
meta_model_df['avg'] = meta_model_df[['xgb','lgb','cat']].mean(axis=1)
for col in meta_model_df.columns:
    print(f'for {col}')
    calculate(meta_model_df[f'{col}'],meta_model_df['accident_risk'])

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor,ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso,ElasticNet, BayesianRidge, SGDRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import cross_val_score

models = {
    'linear': LinearRegression(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'elastic': ElasticNet(),
    # 'bayesian_ridge': BayesianRidge(),
    # 'SGDRegressor':SGDRegressor(),
    # 'rf': RandomForestRegressor(),
    # 'gbr': GradientBoostingRegressor(),
    # 'extra_trees': ExtraTreesRegressor(),
    # 'mlp': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500),
    # 'xgb':XGBRegressor(device='cuda'),
    # 'lgb':LGBMRegressor(device='gpu'),
    # 'cat':CatBoostRegressor(verbose=0,task_type='GPU')
}

rmse_result={}

for name,model in models.items():
    scores = cross_val_score(model,
                    meta_model_df[['xgb','lgb','cat']],
                    meta_model_df['accident_risk'],
                    scoring='neg_root_mean_squared_error',
                    cv=5
                   )
    print(f'for {name} {-scores.mean()}')
    rmse_result[name]=-np.mean(scores)

In [ ]:
meta_model = LinearRegression()
meta_model.fit(
    meta_model_df[['xgb','lgb','cat']],
    meta_model_df['accident_risk']
)

## hypertuning meta model

In [ ]:
def objective(trial):
    params={
        'alpha': trial.suggest_float('alpha', 1e-5, 100, log=True),
        'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False]),
        'solver': trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'lsqr', 'sag', 'saga'])
    }
    scores = cross_val_score(
        Ridge(**params),
        meta_model_df[['xgb','lgb','cat']],
        meta_model_df['accident_risk'],
        cv=5,
        scoring='neg_root_mean_squared_error').mean()

    return -scores


study = optuna.create_study(study_name='Ridge',direction='minimize')
study.optimize(
    objective,
    n_trials=200,
    show_progress_bar=True,
    n_jobs=1
)

In [ ]:
study.best_params

In [ ]:
from optuna.visualization.matplotlib import plot_param_importances, plot_optimization_history
import matplotlib.pyplot as plt

plot_param_importances(study)
plt.show()

plot_optimization_history(study)
plt.show()

In [ ]:
meta_model = Ridge(
    alpha=0.01697754790562364,
    fit_intercept=True,
    solver='sag'
)
meta_model.fit(
    meta_model_df[['xgb','lgb','cat']],
    meta_model_df['accident_risk']
)

# Prediction

In [ ]:
X = feature_engg(test,False)
final_df = pd.DataFrame()

for name,model in final_model.items():
    final_df[f'{name}'] = model.predict(X)

In [ ]:
pred_result = pd.DataFrame(meta_model.predict(final_df))
pred_result.columns = ['accident_risk']

final_result = pd.concat([test['id'],pred_result['accident_risk']],axis=True)
final_result.to_csv('./base_hypertune_with_ridge_hypertune.csv',index=False)

In [ ]:
final_result